# LLM Response Generation

Generates responses to all 600 benchmark complaints from three frontier models under an identical neutral system prompt.

| Model | Provider | Calls |
|-------|----------|-------|
| GPT-4o | OpenAI | 600 |
| Claude 3.5 Sonnet | Anthropic | 600 |
| Gemini 1.5 Flash | Google | 600 |

Checkpoints every 50 rows — safe to interrupt and resume.

In [10]:
import os, time, csv
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from dotenv import load_dotenv

load_dotenv("../.env")

oai = OpenAI(api_key=os.environ["OPEN_AI_API_KEY"])
ant = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
gem = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

SYSTEM_PROMPT = (
    "You are a customer support agent. "
    "Respond to the following customer message in one to three sentences."
)
print("All clients ready")

All clients ready


In [11]:
bench = pd.read_csv("./data/benchmark_600.csv", engine="python")
print(f"Loaded {len(bench)} rows")
bench[["pair_id", "domain", "fis_bin", "customer_text"]].head()

Loaded 600 rows


,pair_id,domain,fis_bin,customer_text
0,427468,airline,low,@AmericanAir Refund issue - waiting 9/13/17. ...
1,2561739,airline,low,Such fun: sunset from 400+ft up! (@ British Ai...
2,669604,airline,low,"@SouthwestAir And wait, we’re also charging fo..."
3,712118,airline,low,@Delta Thank you CX5 Team. Continue to Inspire...
4,266956,airline,low,Hey @Delta - can you help me with a same day c...


In [12]:
def call_gpt4o(complaint: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            r = oai.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": complaint},
                ],
                temperature=0,
            )
            return r.choices[0].message.content.strip()
        except Exception:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt)


def call_gemini(complaint: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            r = gem.models.generate_content(
                model="gemini-3-flash-preview",
                contents=f"{SYSTEM_PROMPT}\n\n{complaint}",
            )
            return r.text.strip()
        except Exception:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt)


print("Generation functions defined (GPT-4o + Gemini)")

Generation functions defined (GPT-4o + Gemini)


In [13]:
# See the "Append Claude responses" cell at the bottom to add claude_response to an existing run.

def call_claude(complaint: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            r = ant.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=256,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": complaint}],
            )
            return r.content[0].text.strip()
        except Exception:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt)

In [14]:
OUT = "./data/llm_responses.csv"
CHECKPOINT = 50

if os.path.exists(OUT):
    done = pd.read_csv(OUT, engine="python")
    done_ids = set(done["pair_id"])
    rows = done.to_dict("records")
    print(f"Resuming — {len(done)} rows already done")
else:
    done_ids = set()
    rows = []

for _, record in bench.iterrows():
    if record["pair_id"] in done_ids:
        continue

    row = {
        "pair_id":        record["pair_id"],
        "domain":         record["domain"],
        "fis_bin":        record["fis_bin"],
        "customer_text":  record["customer_text"],
        "agent_text":     record["agent_text"],
        "gpt4o_response": call_gpt4o(record["customer_text"]),
        "gemini_response": call_gemini(record["customer_text"]),
    }
    rows.append(row)

    if len(rows) % CHECKPOINT == 0:
        pd.DataFrame(rows).to_csv(OUT, index=False, quoting=csv.QUOTE_ALL)
        print(f"  checkpoint — {len(rows)} rows saved")

pd.DataFrame(rows).to_csv(OUT, index=False, quoting=csv.QUOTE_ALL)
print(f"Done — {len(rows)} rows saved to {OUT}")

  checkpoint — 50 rows saved
  checkpoint — 100 rows saved
  checkpoint — 150 rows saved
  checkpoint — 200 rows saved
  checkpoint — 250 rows saved
  checkpoint — 300 rows saved
  checkpoint — 350 rows saved
  checkpoint — 400 rows saved
  checkpoint — 450 rows saved
  checkpoint — 500 rows saved
  checkpoint — 550 rows saved
  checkpoint — 600 rows saved
Done — 600 rows saved to ./data/llm_responses.csv


In [15]:
responses = pd.read_csv(OUT, engine="python")
print(f"Total rows: {len(responses)}")
print(f"\nMissing values:")
print(responses[["gpt4o_response", "gemini_response"]].isna().sum())
print(f"\nSample row:")
print(responses[["customer_text", "gpt4o_response", "gemini_response"]].iloc[0].to_string())

Total rows: 600

Missing values:
gpt4o_response     0
gemini_response    0
dtype: int64

Sample row:
customer_text      @AmericanAir Refund issue - waiting 9/13/17.  ...
gpt4o_response     I'm sorry to hear about the ongoing issue with...
gemini_response    We sincerely apologize for the delay and the f...


## Append Claude responses

Run this section once Claude API is available. Requires `call_claude` to be defined (uncomment the cell above first).
Adds a `claude_response` column to the existing `llm_responses.csv` in a single pass.

In [16]:
# responses = pd.read_csv(OUT, engine="python")

# if "claude_response" in responses.columns:
#     todo = responses[responses["claude_response"].isna()]
#     print(f"Resuming — {len(responses) - len(todo)} already scored, {len(todo)} remaining")
# else:
#     responses["claude_response"] = None
#     todo = responses
#     print(f"Starting fresh — {len(todo)} rows to score")

# for idx, row in todo.iterrows():
#     responses.at[idx, "claude_response"] = call_claude(row["customer_text"])
#     if (idx + 1) % CHECKPOINT == 0:
#         responses.to_csv(OUT, index=False, quoting=csv.QUOTE_ALL)
#         filled = responses["claude_response"].notna().sum()
#         print(f"  checkpoint — {filled} claude responses saved")

# responses.to_csv(OUT, index=False, quoting=csv.QUOTE_ALL)
# print(f"Done — claude_response column complete ({responses['claude_response'].notna().sum()} rows)")